In [46]:
from typing import List
import os
import matplotlib.pyplot as plt
import numpy as np

In [47]:
# ----------------------
# 在此处手动配置数据（用户自行替换）
# parameter 可以是字符串或数字，长度必须与下面三个数组相同

def get_dataList(option):
    if option == 'gradient_loss':
        parameter: List = [0, 1, 3, 5, 10, 20, 30]
        psnr: List[float] = [20.069732475788, 19.921589817655, 19.962562663304, 20.052168828054, 20.032790789288, 20.115088174657, 20.100315154100]
        ssim: List[float] = [0.6428149316182, 0.6404514837854, 0.6422832874971, 0.6428355847507, 0.6435913890191, 0.6445968224006, 0.6453405508001]
        lpips: List[float] = [0.4239501339961, 0.4218084766737, 0.4281946527326, 0.4305711434016, 0.4273177637926, 0.4310304971305, 0.4300452782190]
        title: str = "Gradient Loss"
    elif option == 'perceptual_loss':
        parameter: List = [0, 0.5, 1, 2, 3, 5, 10, 20]
        psnr: List[float] = [20.069732475788, 20.005445891584, 20.085632319214, 20.053150146837, 20.061760320269, 20.059592377053, 19.979334102806, 19.988234057723]
        ssim: List[float] = [0.6428149316182, 0.6409526722569, 0.6392351066879, 0.6370076953766, 0.6344624440987, 0.6313230254784, 0.6233622725314, 0.6155456462972]
        lpips: List[float] = [0.4239501339961, 0.4113136557794, 0.4053262314802, 0.3967384733534, 0.3944751938287, 0.3869503775779, 0.3962489079481, 0.3879546846136]
        title: str = "Perceptual Loss"
    elif option == 'FM':
        parameter: List = [0, 0.5, 1, 2, 3, 5, 10]
        psnr: List[float] = [20.069732475788, 20.049597138464, 20.029095021027, 19.872063563658, 19.942811687571, 19.982909365980, 18.478790349256]
        ssim: List[float] = [0.6428149316182, 0.6432250833575, 0.6420976444506, 0.6372561499121, 0.6412631581978, 0.6380725364305, 0.5750972722480]
        lpips: List[float] = [0.4239501339961, 0.4244695540863, 0.4226833904307, 0.4041979453788, 0.4164464581039, 0.3854115214720, 0.3635500184698]
        title: str = "Feature Matching Loss"
    return parameter, psnr, ssim, lpips, title

# 可选: gradient_loss、FM、perceptual_loss
parameter, psnr, ssim, lpips, title = get_dataList('FM')
# ----------------------

In [48]:
def _validate_lengths():
    n = len(parameter)
    if not (len(psnr) == n and len(ssim) == n and len(lpips) == n):
        raise ValueError("参数长度不一致：`parameter`, `psnr`, `ssim`, `lpips` 必须等长")


def plot_metric(y, ylabel: str, title: str, filename: str = None, show: bool = True):
    _validate_lengths()
    x = list(range(len(parameter)))  # 等间距 x 位置

    plt.figure(figsize=(8, 4))
    plt.plot(x, y, marker="o", linewidth=2)
    plt.xticks(x, parameter)
    plt.xlabel("parameter")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=200)
        print(f"Saved plot to {filename}")
    if show:
        plt.show()
    plt.close()

def plot_combined(parameter, psnr, ssim, lpips, transform_psnr='log', align=False, separate_axis=False, save=False, out_dir='plots', filename='combined.png', show=True):
    """在一张图中绘制三条曲线，并提供多种数值变换以增强可视化。
    参数：
    - parameter: x 轴标签列表
    - psnr, ssim, lpips: 数值列表
    - transform_psnr: 'log'|'minmax'|None，用于对 PSNR 的可选变换（仅在 align=False 时应用）
    - align: bool，若 True 对三条曲线都做 min-max 归一化到 [0,1] 以便比较趋势
    - separate_axis: bool，若 True 将 PSNR 绘制到左轴，SSIM/LPIPS 绘制到右轴
    - save: 是否保存到文件
    - out_dir: 保存目录
    - filename: 保存文件名
    - show: 是否弹窗显示
    """
    # 长度检查
    n = len(parameter)
    if not (len(psnr) == n and len(ssim) == n and len(lpips) == n):
        raise ValueError('parameter/psnr/ssim/lpips 长度必须相同')

    x = list(range(n))
    psnr_arr = np.array(psnr, dtype=float)
    ssim_arr = np.array(ssim, dtype=float)
    lpips_arr = np.array(lpips, dtype=float)

    # 如果选择对齐，则对三条曲线进行 min-max 归一化（0-1），更能比较趋势
    if align:
        def minmax(a):
            a = np.array(a, dtype=float)
            amin, amax = a.min(), a.max()
            return (a - amin) / (amax - amin) if amax > amin else a * 0
        y_psnr = minmax(psnr_arr)
        y_ssim = minmax(ssim_arr)
        y_lpips = minmax(lpips_arr)
        y_label = 'Normalized (0-1)'
        psnr_label = 'PSNR (norm)'
    else:
        # 不对齐时对 PSNR 做指定的变换以减少数量级差距
        if transform_psnr == 'log':
            y_psnr = np.log10(psnr_arr)
            psnr_label = 'PSNR (log10)'
        elif transform_psnr == 'minmax':
            pmin, pmax = psnr_arr.min(), psnr_arr.max()
            y_psnr = (psnr_arr - pmin) / (pmax - pmin) if pmax > pmin else psnr_arr * 0
            psnr_label = 'PSNR (min-max)'
        else:
            y_psnr = psnr_arr
            psnr_label = 'PSNR'
        y_ssim = ssim_arr
        y_lpips = lpips_arr
        y_label = 'Value'

    plt.figure(figsize=(10, 5))
    if separate_axis:
        fig, ax1 = plt.subplots(figsize=(10,5))
        ax2 = ax1.twinx()
        ax1.plot(x, y_psnr, marker='o', linewidth=2, color='C0', label=psnr_label)
        ax2.plot(x, y_ssim, marker='s', linewidth=2, color='C1', label='SSIM')
        ax2.plot(x, y_lpips, marker='^', linewidth=2, color='C2', label='LPIPS')
        ax1.set_xlabel('parameter')
        ax1.set_ylabel(psnr_label)
        ax2.set_ylabel('SSIM / LPIPS')
        ax1.set_xticks(x)
        ax1.set_xticklabels(parameter)
        ax1.grid(True, linestyle='--', alpha=0.3)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
    else:
        plt.plot(x, y_psnr, marker='o', linewidth=2, color='C0', label=psnr_label)
        plt.plot(x, y_ssim, marker='s', linewidth=2, color='C1', label='SSIM')
        plt.plot(x, y_lpips, marker='^', linewidth=2, color='C2', label='LPIPS')
        plt.xticks(x, parameter)
        plt.xlabel('parameter')
        plt.ylabel(y_label)
        plt.title(title)
        plt.grid(True, linestyle='--', alpha=0.3)
        plt.legend(loc='best')

    plt.tight_layout()

    if save:
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, filename)
        plt.savefig(out_path, dpi=200)
        print(f'Saved combined plot to {out_path}')
    if show:
        plt.show()
    plt.close()

def main(save: bool = False, out_dir: str = "plots"):
    if save:
        os.makedirs(out_dir, exist_ok=True)
        plot_metric(psnr, "PSNR", "PSNR vs parameter", os.path.join(out_dir, "psnr.png"), show=False)
        plot_metric(ssim, "SSIM", "SSIM vs parameter", os.path.join(out_dir, "ssim.png"), show=False)
        plot_metric(lpips, "LPIPS", "LPIPS vs parameter", os.path.join(out_dir, "lpips.png"), show=False)
        print(f"All plots saved to {out_dir}/")
    else:
        plot_metric(psnr, "PSNR", "PSNR vs parameter")
        plot_metric(ssim, "SSIM", "SSIM vs parameter")
        plot_metric(lpips, "LPIPS", "LPIPS vs parameter")

In [49]:
# 定位仓库根目录（查找包含 results 或 metrics 的上层目录）
def _find_repo_root(marker_dirs=('results','metrics','imgs')):
    cur = os.path.abspath(os.getcwd())
    while True:
        for d in marker_dirs:
            if os.path.isdir(os.path.join(cur, d)):
                return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            return os.getcwd()
        cur = parent

In [50]:
repo_root = _find_repo_root()
out_dir = os.path.join(repo_root, 'plots')
out_dir = os.path.join(out_dir, title.replace(" ", "_").lower())

print(f'Plots will be saved to: {out_dir}')
# 三张图分别绘制
main(save=True, out_dir=out_dir)
# 使用 align=True 对三条曲线做 min-max 归一化，使趋势更易比较
plot_combined(parameter, psnr, ssim, lpips, align=True, separate_axis=False, save=True, out_dir=out_dir, filename='combined.png', show=False)
print('Done: combined plot saved.')

Plots will be saved to: /root/autodl-tmp/pytorch-CycleGAN-and-pix2pix/plots/feature_matching_loss
Saved plot to /root/autodl-tmp/pytorch-CycleGAN-and-pix2pix/plots/feature_matching_loss/psnr.png
Saved plot to /root/autodl-tmp/pytorch-CycleGAN-and-pix2pix/plots/feature_matching_loss/ssim.png
Saved plot to /root/autodl-tmp/pytorch-CycleGAN-and-pix2pix/plots/feature_matching_loss/lpips.png
All plots saved to /root/autodl-tmp/pytorch-CycleGAN-and-pix2pix/plots/feature_matching_loss/
Saved combined plot to /root/autodl-tmp/pytorch-CycleGAN-and-pix2pix/plots/feature_matching_loss/combined.png
Done: combined plot saved.
